# 04 — Differential expression by Alzheimer's pathology severity

This is the payoff notebook: does gene expression within a specific cell
type differ between donors with more vs. less severe AD neuropathology?

## Concepts before code

**ADNC (Overall AD neuropathological Change).** A standardized
neuropathological staging system combining amyloid-beta plaque burden
(Thal phase), tau tangle spread (Braak stage), and neuritic plaque density
(CERAD score) into a single ordinal severity category: **Not AD < Low <
Intermediate < High**. It's assigned postmortem by a neuropathologist
examining brain tissue directly — it is *not* the same as clinical
dementia status (a donor can have high pathology with little clinical
impairment, or vice versa; SEA-AD deliberately includes such "resilient" or
"susceptible" cases).

**Why compare High vs. Not-AD/Low, and drop Intermediate?** We only have 8
donors total, split roughly 2 per ADNC category. Comparing all 4 ordered
categories with ~2 donors each would be statistically underpowered and
noisy. Comparing the two *extremes* (High pathology vs. Not AD/Low
pathology, dropping the ambiguous Intermediate donors) is what this sample
size can actually support — the same rationale used in the companion
R/Seurat project. This still leaves us with a **small-n comparison** (a
handful of donors per group), so we treat any findings as hypothesis-
generating, not confirmed biology.

**Why restrict to one cell type?** Pooling all cell types together in one
differential expression test conflates two very different signals: (1)
genuine within-cell-type expression changes with disease, and (2) shifts in
cell type *proportions* (e.g. fewer neurons, more reactive microglia, in
more severely affected tissue — itself a real and interesting AD signature,
but a different question). Running DE **within** one well-populated cell
type isolates question (1).

**Pseudobulk vs. per-nucleus testing.** The statistically ideal approach
sums counts per donor (into one "pseudobulk" profile per donor per cell
type) before testing, because nuclei from the same donor aren't independent
samples — treating each nucleus as an independent observation (as
`rank_genes_groups` does by default) inflates apparent significance. We'll
run the simple per-nucleus test first as a worked example (fast, familiar),
then explicitly show why a pseudobulk view changes the picture — this is
an important, easy-to-miss statistical trap in single-cell DE.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
plt.rcParams["figure.dpi"] = 100

PROCESSED_DIR = "/Users/ugobruzadinnunes/Documents/GitHub/sea-ad-python-scanpy/data/processed"
DONOR_META = "/Users/ugobruzadinnunes/Documents/GitHub/sea-ad-multiome-cortex/results/donor_metadata.csv"

adata = sc.read_h5ad(f"{PROCESSED_DIR}/adata_annotated.h5ad")
print(f"Loaded: {adata.n_obs:,} nuclei x {adata.n_vars:,} genes (HVG subset); adata.raw has {adata.raw.n_vars:,} genes")

Loaded: 152,212 nuclei x 2,000 genes (HVG subset); adata.raw has 36,601 genes


## Attach donor-level pathology metadata

In [2]:
donor_meta = pd.read_csv(DONOR_META)
donor_meta = donor_meta.rename(columns={"Donor ID": "donor"}).set_index("donor")
print(donor_meta[["Overall AD neuropathological Change", "Braak", "Age at Death", "Sex"]])

adnc_map = donor_meta["Overall AD neuropathological Change"]
adata.obs["ADNC"] = adata.obs["donor"].map(adnc_map).astype("category")

print("\nNuclei per ADNC group (pooled across all cell types):")
print(adata.obs["ADNC"].value_counts())
print("\nDonors per ADNC group:")
print(donor_meta["Overall AD neuropathological Change"].value_counts())

           Overall AD neuropathological Change      Braak  Age at Death  \
donor                                                                     
H20.33.001                                 Low   Braak IV            82   
H20.33.002                              Not AD   Braak IV            97   
H20.33.004                                High    Braak V            86   
H20.33.005                        Intermediate   Braak IV            99   
H20.33.012                                 Low  Braak III            91   
H20.33.013                        Intermediate   Braak IV            94   
H20.33.017                                High   Braak VI            69   
H21.33.003                              Not AD    Braak 0            78   

               Sex  
donor               
H20.33.001    Male  
H20.33.002  Female  
H20.33.004    Male  
H20.33.005  Female  
H20.33.012  Female  
H20.33.013    Male  
H20.33.017    Male  
H21.33.003    Male  

Nuclei per ADNC group (pooled across a

## Define the High vs. Low/Not-AD contrast, excluding Intermediate

In [3]:
adata.obs["pathology_group"] = adata.obs["ADNC"].map(
    {"Not AD": "Low_pathology", "Low": "Low_pathology",
     "Intermediate": "Intermediate", "High": "High_pathology"}
).astype("category")

contrast = adata[adata.obs["pathology_group"].isin(["Low_pathology", "High_pathology"])].copy()
print(contrast.obs.groupby(["pathology_group", "donor"], observed=True).size())

pathology_group  donor     
High_pathology   H20.33.004    20297
                 H20.33.017    14420
Low_pathology    H20.33.001    19160
                 H20.33.002    19548
                 H20.33.012    16357
                 H21.33.003    18856
dtype: int64


## Choosing a well-populated cell type

Excitatory neurons are almost always the largest population in cortex —
let's confirm that here and use them for the worked example.

In [4]:
print(contrast.obs["cell_type"].value_counts())

TARGET_CELL_TYPE = "Excitatory"
sub = contrast[contrast.obs["cell_type"] == TARGET_CELL_TYPE].copy()
print(f"\n{TARGET_CELL_TYPE} nuclei in contrast: {sub.n_obs:,}")
print(sub.obs.groupby(["pathology_group", "donor"], observed=True).size())

cell_type
Excitatory         52454
Inhibitory         28265
Vascular           10716
Astrocyte           5806
Oligodendrocyte     5413
Microglia           3192
OPC                 2792
Name: count, dtype: int64



Excitatory nuclei in contrast: 52,454
pathology_group  donor     
High_pathology   H20.33.004    10219
                 H20.33.017     6171
Low_pathology    H20.33.001     9549
                 H20.33.002    10163
                 H20.33.012     8326
                 H21.33.003     8026
dtype: int64


## Step 1 — worked example: per-nucleus differential expression

The fast, familiar approach: treat each nucleus as an independent sample
and run the same Wilcoxon test as notebook 03, now grouping by pathology
instead of cluster.

In [5]:
sc.tl.rank_genes_groups(
    sub, groupby="pathology_group", groups=["High_pathology"], reference="Low_pathology",
    method="wilcoxon", use_raw=True, key_added="de_pernucleus",
)

de_pernucleus = sc.get.rank_genes_groups_df(sub, group="High_pathology", key="de_pernucleus")
de_pernucleus = de_pernucleus.sort_values("pvals_adj")
print(f"Genes with adjusted p < 0.05 (per-nucleus test): {(de_pernucleus['pvals_adj'] < 0.05).sum()} / {len(de_pernucleus)}")
de_pernucleus.head(15)

Genes with adjusted p < 0.05 (per-nucleus test): 10422 / 36601


,names,scores,logfoldchanges,pvals,pvals_adj
0,ARL17B,116.153900,1.906832,0.0,0.0
36569,LINC01115,-42.420380,-1.957288,0.0,0.0
36568,FAM106A,-42.302601,-0.611518,0.0,0.0
36567,GRIA1,-41.936337,-0.311151,0.0,0.0
36566,SORBS2,-41.812107,-0.381997,0.0,0.0
36565,NEDD4L,-41.775307,-0.303132,0.0,0.0
36564,SHC3,-41.682903,-0.473347,0.0,0.0
36563,CHL1-AS2,-41.559143,-1.123149,0.0,0.0
36562,KIAA1211L,-41.195145,-0.363569,0.0,0.0
36561,PTPN14,-40.929874,-0.632785,0.0,0.0


## Step 2 — the pseudobulk check

Now collapse to one profile **per donor** (sum raw counts across all
`TARGET_CELL_TYPE` nuclei belonging to that donor, then normalize), so each
donor contributes exactly one data point — the statistically defensible
unit of replication here, since donors (not nuclei) are the independent
samples.

In [6]:
donor_ids_in_contrast = sub.obs["donor"].unique()
pseudobulk_rows = []
for donor in donor_ids_in_contrast:
    mask = sub.obs["donor"] == donor
    summed_counts = np.asarray(sub.raw[mask].X.expm1().sum(axis=0)).flatten()
    # .raw.X is log1p-normalized; expm1 undoes the log for a fairer sum, then
    # we renormalize the pseudobulk profile below.
    pseudobulk_rows.append(summed_counts)

pseudobulk = pd.DataFrame(
    np.array(pseudobulk_rows), index=donor_ids_in_contrast, columns=sub.raw.var_names
)
# CPM-style normalization + log1p, per pseudobulk sample.
pseudobulk_norm = np.log1p(pseudobulk.div(pseudobulk.sum(axis=1), axis=0) * 1e4)

pathology_by_donor = sub.obs.drop_duplicates("donor").set_index("donor")["pathology_group"]
pseudobulk_norm["pathology_group"] = pathology_by_donor.reindex(pseudobulk_norm.index)

print(pseudobulk_norm[["pathology_group"]])
print(f"\nPseudobulk matrix: {pseudobulk_norm.shape[0]} donors x {pseudobulk_norm.shape[1]-1} genes")

index      pathology_group
H20.33.001   Low_pathology
H20.33.002   Low_pathology
H20.33.004  High_pathology
H20.33.012   Low_pathology
H20.33.017  High_pathology
H21.33.003   Low_pathology

Pseudobulk matrix: 6 donors x 36601 genes


With only 2-4 donors per group, we can't run a formal per-gene test with
any real power — instead, look directly at whether the top per-nucleus
hits from Step 1 hold up as a *consistent direction of difference* at the
donor/pseudobulk level (a much simpler, more honest bar given this sample
size).

In [7]:
top_genes = de_pernucleus.head(15)["names"].tolist()
pb_check = pseudobulk_norm[top_genes + ["pathology_group"]].copy()
pb_check_display = pb_check.set_index("pathology_group")
pb_check_display

index,ARL17B,LINC01115,FAM106A,GRIA1,SORBS2,NEDD4L,SHC3,CHL1-AS2,KIAA1211L,PTPN14,GABRB3,TNR,RPH3A,PRKCE,FLRT2
pathology_group,,,,,,,,,,,,,,,
Low_pathology,0.054385,0.168110,0.661536,1.923267,2.200271,1.628795,0.884002,0.235193,1.560430,0.587792,2.230635,1.638428,0.556830,1.661962,2.256321
Low_pathology,0.182563,0.144864,0.727065,1.815360,2.073016,1.593752,1.006757,0.174589,1.673422,0.530322,2.234865,1.626648,0.528754,1.757895,2.150522
High_pathology,0.937300,0.016643,0.490393,1.638952,1.882163,1.468355,0.847796,0.122449,1.481361,0.440749,2.076518,1.444674,0.450954,1.604989,1.895226
Low_pathology,1.237363,0.192336,0.613113,1.820172,2.256901,1.484559,1.192109,0.422953,1.511939,0.537912,2.262902,1.743643,0.818019,1.946068,2.352154
High_pathology,1.168803,0.113072,0.495935,1.731294,1.895081,1.396413,0.742662,0.181992,1.373895,0.346823,2.129266,1.512722,0.456545,1.666600,2.263683
Low_pathology,0.147826,0.240890,0.743579,1.764744,1.947791,1.693041,0.898230,0.341704,1.680530,0.602492,2.103036,1.451125,0.709801,1.746342,2.047323


In [8]:
low_mean = pb_check[pb_check["pathology_group"] == "Low_pathology"][top_genes].mean()
high_mean = pb_check[pb_check["pathology_group"] == "High_pathology"][top_genes].mean()
direction_check = pd.DataFrame({
    "per_nucleus_logFC": de_pernucleus.set_index("names").loc[top_genes, "logfoldchanges"],
    "pseudobulk_low_mean": low_mean,
    "pseudobulk_high_mean": high_mean,
    "pseudobulk_direction_agrees": np.sign(high_mean - low_mean) == np.sign(
        de_pernucleus.set_index("names").loc[top_genes, "logfoldchanges"]
    ),
})
direction_check

,per_nucleus_logFC,pseudobulk_low_mean,pseudobulk_high_mean,pseudobulk_direction_agrees
ARL17B,1.906832,0.405534,1.053051,True
LINC01115,-1.957288,0.186550,0.064857,True
FAM106A,-0.611518,0.686324,0.493164,True
GRIA1,-0.311151,1.830886,1.685123,True
SORBS2,-0.381997,2.119495,1.888622,True
NEDD4L,-0.303132,1.600037,1.432384,True
SHC3,-0.473347,0.995274,0.795229,True
CHL1-AS2,-1.123149,0.293609,0.152221,True
KIAA1211L,-0.363569,1.606580,1.427628,True
PTPN14,-0.632785,0.564630,0.393786,True


In [9]:
n_agree = direction_check["pseudobulk_direction_agrees"].sum()
print(f"Of the top {len(top_genes)} per-nucleus hits, {n_agree} agree in direction at the pseudobulk (donor) level.")
print("Genes that do NOT hold up at pseudobulk level are prime candidates for being driven by one noisy")
print("donor or by nucleus-count non-independence, rather than a real donor-level pathology effect.")

Of the top 15 per-nucleus hits, 15 agree in direction at the pseudobulk (donor) level.
Genes that do NOT hold up at pseudobulk level are prime candidates for being driven by one noisy
donor or by nucleus-count non-independence, rather than a real donor-level pathology effect.


## Honest interpretation

With **2-4 donors per group**, nothing here should be read as a validated
AD signature — that would need a much larger cohort (SEA-AD's full public
release has dozens of donors; this project deliberately works with an
8-donor subset for tractability). What this analysis *can* responsibly
claim:
- A workflow for going from raw counts to a defensible pathology contrast
  within one cell type, including the pseudobulk sanity check that a naive
  per-nucleus test skips.
- A short list of genes worth flagging as hypotheses for a follow-up study
  with more donors — genes whose per-nucleus signal is also directionally
  consistent at the donor level, from the `direction_check` table above.
- A concrete illustration of *why* n=2-4 donor comparisons in single-cell
  AD studies need this kind of scrutiny before being reported as findings.

## Recap

- Merged donor-level ADNC pathology staging onto per-nucleus data via a
  simple column mapping.
- Defined a High-vs-Low/Not-AD contrast, dropping Intermediate donors, to
  match what an 8-donor cohort can actually support.
- Ran DE within one well-populated cell type (excitatory neurons) to avoid
  conflating expression changes with cell-type-proportion shifts.
- Showed the gap between naive per-nucleus significance and a
  donor-level pseudobulk check — and why the latter is the more defensible
  read given so few donors.

Next (optional): **05_atac_intro_bonus.ipynb** — a light look at the paired
ATAC (chromatin accessibility) data.

## Homework

1. **Different cell type.** Rerun Step 1 (per-nucleus `rank_genes_groups`)
   restricted to `cell_type == "Microglia"` instead of Excitatory. Microglia
   are classically implicated in AD neuroinflammation — do you see
   different top genes? (Caveat: microglia are a much smaller population —
   check `sub.obs["cell_type"].value_counts()`-style counts first and keep
   the small-n caveat in mind even more strongly here.)

2. **Write the code yourself:** compute the pseudobulk profile the same way
   as Step 2, but for `TARGET_CELL_TYPE = "Astrocyte"`, and check whether
   `GFAP` (a classic marker of astrocyte reactivity/gliosis in
   neurodegeneration) shows a consistent direction of difference between
   pathology groups.

3. **Interpretation.** Pick one gene from the `direction_check` table above
   whose direction *did* agree between per-nucleus and pseudobulk levels.
   Look up what it does. Does a role in neurodegeneration, synaptic
   function, or neuroinflammation make biological sense, or does it look
   more like a generic stress-response gene?

4. **Open-ended.** Braak stage (tau tangle spread) and ADNC are related but
   not identical. Using `donor_meta`, define an alternative contrast based
   on `Braak` instead of `Overall AD neuropathological Change` (e.g.
   Braak 0-III vs. Braak IV-VI) and see whether the donor groupings — and
   thus your candidate gene list — change.

In [10]:
# Q1 — your code here

In [11]:
# Q2 — your code here

Q3 — your answer here

In [12]:
# Q4 — your code here